# Testy jednostkowe czarnoskrzynkowe

System: NoteDiscovery (nasz fork: LepsyMikolaj3301/NoteDiscovery-qa-testing). Testujemy tylko backend w Pythonie.

Skład: Mikołaj (B1-B15), Paola (B16-B26)

## Jak wybieraliśmy funkcje i przypadki

Wybraliśmy funkcje, które naszym zdaniem są najważniejsze w aplikacji: obsługę tagów, linków i szablonów notatek,
logowanie i klucze API, sprawdzanie ścieżek i nazw plików, udostępnianie notatek i część serwera MCP (przez niego z notatkami łączy się agent AI).

Testy pisaliśmy na podstawie opisu funkcji (docstringów) i dokumentacji projektu w folderze `documentation/`.
W kilku miejscach dokumentacja czegoś nie mówiła i musieliśmy przyjąć założenie albo sprawdzić wartość w kodzie. Takie miejsca są opisane w kolumnie „Uwagi”.

Przypadki dzieliliśmy tak:
- z dokumentacji: przykład podany wprost w docstringu albo dokumentacji,
- poprawne: typowe, poprawne dane,
- błędne: dane niepoprawne, których funkcja nie powinna przyjąć,
- graniczne: wartości na granicy, np. pusty tekst, 0, najmniejsza i największa dozwolona długość.

Testy są w folderze `tests/unit/blackbox/` (pytest). Pod każdą tabelą jest komórka, która uruchamia testy dla tej funkcji.

## Przypadki testowe

### B1. paginate (backend/utils.py)

Funkcja dzieli listę na strony. Jeśli `limit` jest `None`, zwraca wszystko. Można też podać `sort_key` i `sort_reverse`. Wynik ma pola `items`, `total`, `limit`, `offset` i `has_more`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B1.1 | poprawne | lista 10 elementów, limit=None | wszystkie elementy, total=10, has_more=False |  |
| B1.2 | poprawne | limit=3, offset=0 | [0, 1, 2], has_more=True |  |
| B1.3 | graniczne | limit=3, offset=9 | [9], has_more=False | ostatni element |
| B1.4 | graniczne | limit=3, offset=10 | [], has_more=False | offset równy długości listy |
| B1.5 | błędne | limit=3, offset=99 | [], has_more=False |  |
| B1.6 | błędne | limit=3, offset=-5 | [0, 1, 2], has_more=True | w docstringu nie ma nic o ujemnym offsecie, przyjęliśmy że liczy się jak 0 |
| B1.7 | graniczne | limit=0 | [], has_more=True | też nie jest opisane |
| B1.8 | graniczne | limit=10 | wszystkie, has_more=False | limit równy długości listy |
| B1.9 | poprawne | ['b','c','a'], limit=2, sortowanie malejąco | ['c','b'], to_dict(): total=3, limit=2, has_more=True |  |

In [105]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_no_limit_returns_all tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_sort_and_to_dict -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 9 items

tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_no_limit_returns_all PASSED [ 11%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[3-0-expected0-True] PASSED [ 22%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[3-9-expected1-False] PASSED [ 33%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[3-10-expected2-False] PASSED [ 44%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[3-99-expected3-False] PASSED [ 55%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[3--5-expected4-True] PASSED [ 66%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[0-0-expected5-True] PASSED [ 77%]
tests/unit/blackbox/test_utils_export_helpers.py::test_paginate_bounds[10-0-expected6-False] PASSED [ 88%]
tests/unit/blackbox/test_utils_export_helper

### B2. sanitize_filename (backend/utils.py)

Usuwa z nazwy pliku znaki niedozwolone w Windows (`\ / : * ? " < > |`) i znaki sterujące (0x00-0x1f). Polskie znaki mają zostać.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B2.1 | błędne | `a?b.md` i tak samo dla każdego z 9 znaków | znaku nie ma w wyniku, końcówka `.md` zostaje | 9 osobnych przypadków |
| B2.2 | graniczne | nazwa ze znakiem `\x00` i `\x1f` | znak usunięty | pierwszy i ostatni znak sterujący |
| B2.3 | poprawne | `'zażółć.md'` | bez zmian |  |
| B2.4 | graniczne | `''` | `''` |  |
| B2.5 | poprawne | `'noext'` | bez zmian | nazwa bez rozszerzenia |

In [106]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_keeps_unicode_and_edge_cases -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 12 items

tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[\\] PASSED [  8%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[/] PASSED [ 16%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[:] PASSED [ 25%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[*] PASSED [ 33%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[?] PASSED [ 41%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden["] PASSED [ 50%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[<] PASSED [ 58%]
tests/unit/blackbox/test_utils_export_helpers.py::test_sanitize_filename_removes_forbidden[>] PASSED [ 66%]
tests/unit/blackbox/

### B3. get_media_type (backend/utils.py)

Po rozszerzeniu pliku zwraca typ: `image`, `audio`, `video`, `document`, `drawing` albo `None`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B3.1 | poprawne | `'a.png'` | `'image'` |  |
| B3.2 | poprawne | `'a.JPG'` | `'image'` | duże litery w rozszerzeniu |
| B3.3 | poprawne | `'a.mp3'` | `'audio'` |  |
| B3.4 | poprawne | `'a.mp4'` | `'video'` |  |
| B3.5 | poprawne | `'a.pdf'` | `'document'` |  |
| B3.6 | błędne | `'a.xyz'` | None | nieznane rozszerzenie |
| B3.7 | błędne | `'noext'` | None | brak rozszerzenia |

In [107]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 7 items

tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.png-image] PASSED [ 14%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.JPG-image] PASSED [ 28%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.mp3-audio] PASSED [ 42%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.mp4-video] PASSED [ 57%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.pdf-document] PASSED [ 71%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[a.xyz-None] PASSED [ 85%]
tests/unit/blackbox/test_utils_export_helpers.py::test_get_media_type[noext-None] PASSED [100%]

============================== 7 passed in 0.03s ==============================


### B4. path_has_hash (backend/utils.py)

Zwraca `True`, jeśli w nazwie notatki albo folderu jest `#` (ten znak jest używany w linkach do nagłówków).

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B4.1 | poprawne | `'a#b.md'` | True | # w nazwie pliku |
| B4.2 | poprawne | `'dir#x/a.md'` | True | # w nazwie folderu |
| B4.3 | błędne | `'dir/a.md'` | False |  |
| B4.4 | graniczne | `''` | False | pusta ścieżka |

In [108]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_path_has_hash -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 4 items

tests/unit/blackbox/test_utils_export_helpers.py::test_path_has_hash[a#b.md-True] PASSED [ 25%]
tests/unit/blackbox/test_utils_export_helpers.py::test_path_has_hash[dir#x/a.md-True] PASSED [ 50%]
tests/unit/blackbox/test_utils_export_helpers.py::test_path_has_hash[dir/a.md-False] PASSED [ 75%]
tests/unit/blackbox/test_utils_export_helpers.py::test_path_has_hash[-False] PASSED [100%]

============================== 4 passed in 0.03s ==============================


### B5. parse_image_size_spec (backend/export.py)

Odczytuje rozmiar obrazka zapisany jak w Obsidianie, np. `opis|100x200`. W docstringu jest gotowa lista przykładów, więc wzięliśmy je prawie wszystkie. Wymiar 0 oznacza „nie ustawiono”.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B5.1 | z dokumentacji | `'caption'` | ('caption', None, None) |  |
| B5.2 | z dokumentacji | `'caption\|100'` | ('caption', 100, None) |  |
| B5.3 | z dokumentacji | `'caption\|100x200'` | ('caption', 100, 200) |  |
| B5.4 | z dokumentacji | `'100'`, allow_solo=True | ('', 100, None) |  |
| B5.5 | z dokumentacji | `'100x200'`, allow_solo=True | ('', 100, 200) |  |
| B5.6 | błędne | `'100'`, allow_solo=False | ('100', None, None) | liczba traktowana jako opis |
| B5.7 | graniczne | `'c\|0x200'` | ('c', None, 200) | szerokość 0 |
| B5.8 | graniczne | `''` | ('', None, None) |  |
| B5.9 | błędne | `'cap\|abc'` | szerokość i wysokość None | tekst zamiast liczby |

In [109]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec_garbage_has_no_size -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 9 items

tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[caption-False-expected0] PASSED [ 11%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[caption|100-False-expected1] PASSED [ 22%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[caption|100x200-False-expected2] PASSED [ 33%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[100-True-expected3] PASSED [ 44%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[100x200-True-expected4] PASSED [ 55%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[100-False-expected5] PASSED [ 66%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_spec[c|0x200-False-expected6] PASSED [ 77%]
tests/unit/blackbox/test_utils_export_helpers.py::test_parse_image_size_

### B6. attachment_content_disposition (backend/export.py)

Tworzy nagłówek `Content-Disposition` dla pobieranego pliku. Zwykła nazwa idzie do `filename="..."`, a nazwa z np. polskimi znakami do `filename*=utf-8''...`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B6.1 | poprawne | `'note.html'` | `attachment; filename="note.html"` |  |
| B6.2 | poprawne | `'zażółć.html'` | zawiera `filename*=utf-8''` |  |
| B6.3 | błędne | `'a";b.html'` | cudzysłów nie może zamknąć nagłówka |  |

In [110]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_content_disposition_ascii tests/unit/blackbox/test_utils_export_helpers.py::test_content_disposition_non_ascii_and_special -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 2 items

tests/unit/blackbox/test_utils_export_helpers.py::test_content_disposition_ascii PASSED [ 50%]
tests/unit/blackbox/test_utils_export_helpers.py::test_content_disposition_non_ascii_and_special PASSED [100%]

============================== 2 passed in 0.02s ==============================


### B7. strip_frontmatter (backend/export.py)

Usuwa frontmatter, czyli blok między `---` na początku notatki.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B7.1 | poprawne | `'---\ntags: a\n---\nbody'` | `'body'` |  |
| B7.2 | poprawne | `'body'` | `'body'` | brak frontmattera |
| B7.3 | błędne | `'---\ntags: a\nbody'` | bez zmian | brak zamykającego --- |
| B7.4 | błędne | `'text\n---\nmore'` | bez zmian | --- w środku tekstu |

In [111]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_strip_frontmatter -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 1 item

tests/unit/blackbox/test_utils_export_helpers.py::test_strip_frontmatter PASSED [100%]

============================== 1 passed in 0.02s ==============================


### B8. generate_media_placeholder (backend/export.py)

Tworzy zastępczy blok HTML dla audio, wideo i PDF w eksporcie (bo tych plików nie da się osadzić).

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B8.1 | poprawne | typ `audio`, `video`, `document`, nazwa `x.ext` | w HTML jest nazwa `x.ext` | 3 przypadki |
| B8.2 | błędne | nazwa z tagami HTML: `'<b>"x"</b>'` | znaki są zamienione na `&lt;`, `&quot;` itd. | żeby nie dało się wstawić HTML-a |

In [112]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_per_type tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_escapes_alt -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 4 items

tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_per_type[audio] PASSED [ 25%]
tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_per_type[video] PASSED [ 50%]
tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_per_type[document] PASSED [ 75%]
tests/unit/blackbox/test_utils_export_helpers.py::test_media_placeholder_escapes_alt PASSED [100%]

============================== 4 passed in 0.03s ==============================


### B9. _normalize_share_public_origin (backend/main.py)

Z adresu ustawionego w konfiguracji zostawia tylko `schemat://host[:port]`. Dla złego albo pustego adresu zwraca `''`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B9.1 | graniczne | None, `''`, `'  '` | `''` | 3 przypadki |
| B9.2 | poprawne | `'http://example.com'` | bez zmian |  |
| B9.3 | poprawne | `'https://example.com:8443/path?q=1'` | `'https://example.com:8443'` | ścieżka i parametry usunięte |
| B9.4 | błędne | `'ftp://example.com'` | `''` | zły protokół |
| B9.5 | błędne | `'example.com'` | `''` | brak protokołu |
| B9.6 | błędne | `'http://'` | `''` | brak hosta |

In [113]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 8 items

tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[None-] PASSED [ 12%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[-] PASSED [ 25%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[  -] PASSED [ 37%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[http://example.com-http://example.com] PASSED [ 50%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[https://example.com:8443/path?q=1-https://example.com:8443] PASSED [ 62%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[ftp://example.com-] PASSED [ 75%]
tests/unit/blackbox/test_utils_export_helpers.py::test_normalize_share_public_origin[example.com-] PASSED [ 87%]
tests/unit/blackbox/test_utils_export_helpers

### B10. safe_error_message (backend/main.py)

W trybie debug zwraca pełny opis błędu, a normalnie tylko ogólny komunikat dla użytkownika.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B10.1 | poprawne | ValueError('secret'), komunikat 'oops', debug wyłączony | `'oops'` |  |
| B10.2 | poprawne | to samo, debug włączony | `'ValueError: secret'` |  |

In [114]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_utils_export_helpers.py::test_safe_error_message -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 1 item

tests/unit/blackbox/test_utils_export_helpers.py::test_safe_error_message PASSED [100%]

============================== 1 passed in 0.02s ==============================


### B11. Plugin.calculate_stats (plugins/note_stats.py)

Liczy statystyki notatki. Według `PLUGIN_NOTE_STATISTICS.md`: słowa, znaki, nagłówki, bloki kodu, linki (wewnętrzne to pliki `.md` i wikilinki, zewnętrzne to http/https), zadania i czas czytania (200 słów na minutę).

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B11.1 | graniczne | pusta notatka | 0 słów, 0 znaków, 0 zadań, czas czytania co najmniej 1 | z dokumentacji wychodzi 0 minut, a funkcja zwraca 1, na razie to przyjęliśmy |
| B11.2 | poprawne | notatka z nagłówkiem, 2 zadaniami, blokiem kodu i 3 linkami | h1=1, bloki kodu=1, zadania 2 (1 zrobione), linki 3 (2 wewnętrzne, 1 zewnętrzny), wikilinki 1 |  |
| B11.3 | poprawne | 2000 słów i 1 słowo | dla 2000 słów czas jest dłuższy |  |

In [115]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_empty_note tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_counts tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_reading_time_grows_with_words -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 3 items

tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_empty_note PASSED [ 33%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_counts PASSED     [ 66%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_stats_reading_time_grows_with_words PASSED [100%]

============================== 3 passed in 0.03s ==============================


### B12. load_config (mcp_server/config.py)

Wczytuje konfigurację serwera MCP ze zmiennych środowiskowych. Domyślnie adres `http://localhost:8000`, timeout 30, 3 próby. Przy złej konfiguracji ma rzucić `ValueError`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B12.1 | poprawne | brak zmiennych | adres domyślny, brak klucza API |  |
| B12.2 | poprawne | adres `'http://host:1234///'` | `'http://host:1234'` | usunięte ukośniki na końcu |
| B12.3 | błędne | `'not a url'`, `'ftp://host'`, `'localhost:8000'`, `''` | ValueError | 4 przypadki |
| B12.4 | błędne | timeout `abc`, liczba prób `x` | wartości domyślne 30 i 3 | docstring mówi o ValueError, a funkcja bierze wartości domyślne |
| B12.5 | poprawne | bez klucza / z kluczem `k` | brak nagłówka `X-API-Key` / nagłówek `X-API-Key: k` |  |

In [116]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_plugin_and_mcp.py::test_config_defaults tests/unit/blackbox/test_plugin_and_mcp.py::test_config_strips_trailing_slash tests/unit/blackbox/test_plugin_and_mcp.py::test_config_invalid_url tests/unit/blackbox/test_plugin_and_mcp.py::test_config_bad_numbers_fall_back tests/unit/blackbox/test_plugin_and_mcp.py::test_config_headers -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 8 items

tests/unit/blackbox/test_plugin_and_mcp.py::test_config_defaults PASSED  [ 12%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_strips_trailing_slash PASSED [ 25%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_invalid_url[not a url] PASSED [ 37%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_invalid_url[ftp://host] PASSED [ 50%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_invalid_url[localhost:8000] PASSED [ 62%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_invalid_url[] PASSED [ 75%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_bad_numbers_fall_back PASSED [ 87%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_config_headers PASSED   [100%]

============================== 8 passed in 0.03s ==============================


### B13. MCPServer._validate_path (mcp_server/server.py)

Sprawdza, czy ścieżka podana przez agenta AI jest bezpieczna. Zwraca parę `(czy_ok, komunikat)`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B13.1 | błędne | `''` | False + komunikat |  |
| B13.2 | błędne | `'../x.md'`, `'a/../b.md'` | False + komunikat | wyjście poza folder |
| B13.3 | błędne | `'/abs.md'`, `'\\abs.md'`, `'C:\\x.md'` | False + komunikat | ścieżki bezwzględne |
| B13.4 | błędne | `'a\x00.md'` | False + komunikat | bajt zerowy |
| B13.5 | poprawne | `'folder/note.md'` | True |  |

In [117]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_accepts -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 8 items

tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[] PASSED [ 12%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[../x.md] PASSED [ 25%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[a/../b.md] PASSED [ 37%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[/abs.md] PASSED [ 50%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[\\abs.md] PASSED [ 62%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[C:\\x.md] PASSED [ 75%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_rejects[a\x00.md] PASSED [ 87%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_validate_path_accepts PASSED [100%]

============================== 8 passed in 0.03s ==============================


### B14. MCPServer._format_response (mcp_server/server.py)

Zamienia odpowiedź z API na tekst dla agenta AI.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B14.1 | błędne | odpowiedź z błędem `boom` | `'Error: boom'` |  |
| B14.2 | graniczne | sukces bez danych | `'Success (no data)'` |  |
| B14.3 | poprawne | sukces z danymi `{'t': 'zażółć'}` | poprawny JSON, polskie znaki bez zmian |  |

In [118]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_plugin_and_mcp.py::test_format_response -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 1 item

tests/unit/blackbox/test_plugin_and_mcp.py::test_format_response PASSED  [100%]

============================== 1 passed in 0.03s ==============================


### B15. get_tool_names, get_tool_by_name, TOOLS (mcp_server/tools.py)

Lista narzędzi MCP i wyszukiwanie narzędzia po nazwie. Według protokołu MCP każde narzędzie ma `name`, `description` i `inputSchema`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B15.1 | błędne | `'nope'` | None |  |
| B15.2 | poprawne | pierwsza nazwa z listy | zwraca definicję |  |
| B15.3 | poprawne | wszystkie narzędzia | nazwy się nie powtarzają, każde ma wymagane pola |  |
| B15.4 | poprawne | wszystkie narzędzia | serwer ma metodę `_tool_<nazwa>` dla każdego | tę konwencję nazw podejrzeliśmy w kodzie |

In [119]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_plugin_and_mcp.py::test_tool_lookup tests/unit/blackbox/test_plugin_and_mcp.py::test_tool_schema_sanity -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 2 items

tests/unit/blackbox/test_plugin_and_mcp.py::test_tool_lookup PASSED      [ 50%]
tests/unit/blackbox/test_plugin_and_mcp.py::test_tool_schema_sanity PASSED [100%]

============================== 2 passed in 0.03s ==============================


### B16. parse_tags (backend/utils.py)

Notatki w NoteDiscovery to zwykłe pliki tekstowe w formacie Markdown. Na samej górze pliku można dopisać tzw. **frontmatter**, czyli kilka linii z informacjami o notatce, zamkniętych między dwiema liniami `---`. Jedną z tych informacji są **tagi**: słowa kluczowe, po których potem łatwo znaleźć notatkę. Przykład:

```
---
tags: [python, tutorial]
---
Treść notatki
```

Funkcja dostaje cały tekst notatki i zwraca listę jej tagów. Tagi można wpisać w jednej linii w nawiasach kwadratowych (jak wyżej) albo każdy w osobnej linii z myślnikiem na początku. Tagi w wyniku mają być pisane małymi literami i każdy ma się pojawić tylko raz. Tag może mieć też podkategorię po ukośniku, np. `projekt/backend`.

W tabeli `\n` oznacza przejście do nowej linii.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B16.1 | z dokumentacji | tagi w jednej linii: `'---\ntags: [python, tutorial]\n---\nbody'` | `python`, `tutorial` |  |
| B16.2 | z dokumentacji | te same tagi, każdy w osobnej linii z myślnikiem: `'---\ntags:\n  - python\n  - tutorial\n---\nbody'` | `python`, `tutorial` |  |
| B16.3 | poprawne | jeden tag bez nawiasów: `tags: python` | `python` |  |
| B16.4 | poprawne | ten sam tag wpisany dwa razy: `[a, a, b]`, i to samo w wersji z myślnikami (dwa razy `a`) | `a`, `b` oraz `a` | każdy tag tylko raz |
| B16.5 | poprawne | `[Python, PYTHON]` oraz `[Python]` | `python` | wielkie litery zamieniane na małe, więc to jeden tag |
| B16.6 | poprawne | tag z podkategorią: `[project/backend]` | `project/backend` | ukośnik zostaje |
| B16.7 | poprawne | przed tagami jest inna informacja: `'---\ntitle: x\ntags: [a]\n---\nbody'` | `a` | tagi nie muszą być w pierwszej linii |
| B16.8 | graniczne | lista z myślnikami przerwana inną informacją: `'---\ntags:\n  - a\ntitle: x\n  - b\n---\nbody'` | tylko `a` | lista tagów kończy się na `title`, więc `b` nie jest już tagiem |
| B16.9 | błędne | pusty tekst; notatka bez frontmattera; `tags: [a]` wpisane w treści notatki, a nie we frontmatterze | pusta lista | tagi liczą się tylko we frontmatterze |
| B16.10 | błędne | frontmatter bez zamykającej linii `---` | pusta lista | niedokończony frontmatter jest ignorowany |
| B16.11 | graniczne | frontmatter bez tagów; pusta lista `tags: []`; samo `tags:` bez niczego | pusta lista |  |
| B16.12 | poprawne | plik zapisany na Windowsie (tam koniec linii to `\r\n` zamiast `\n`) | `a`, `b` | wynik ma być taki sam jak na innych systemach |
| B16.13 | poprawne | tagi w cudzysłowach: `tags: ["a", "b"]` oraz `tags: ['a', 'b']` | `a`, `b` (bez cudzysłowów) | **test nie przechodzi, znaleziony błąd** (opis na końcu) |

In [120]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_returns_empty tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_crlf tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_quoted_values -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 20 items

tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags: [python, tutorial]\n---\nbody-expected0] PASSED [  5%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags:\n  - python\n  - tutorial\n---\nbody-expected1] PASSED [ 10%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags: python\n---\nbody-expected2] PASSED [ 15%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags: [a, a, b]\n---\nbody-expected3] PASSED [ 20%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags:\n  - a\n  - a\n---\nbody-expected4] PASSED [ 25%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_extracts[---\ntags: [Python, PYTHON]\n---\nbody-expected5] PASSED [ 30%]
tests/unit/blackbox/test_tags_templates_links.py::test_parse

### B17. apply_template_placeholders (backend/utils.py)

Nową notatkę można utworzyć z **szablonu**, czyli gotowego wzoru notatki. W szablonie mogą być znaczniki w podwójnych nawiasach klamrowych, które przy tworzeniu notatki zamieniają się na aktualne wartości. Na przykład `{{date}}` zamienia się na dzisiejszą datę, a `{{title}}` na nazwę notatki. Tę zamianę robi ta funkcja.

Dostępne znaczniki:
- `{{date}}` – data, `{{time}}` – godzina, `{{datetime}}` – data i godzina,
- `{{timestamp}}` – czas zapisany jako liczba sekund od 1 stycznia 1970 r.,
- `{{year}}`, `{{month}}`, `{{day}}` – rok, miesiąc, dzień,
- `{{title}}` – nazwa notatki, `{{folder}}` – folder, w którym leży notatka.

Po dwukropku można podać własny sposób zapisu daty, np. `{{date:%d/%m/%Y}}` daje `06/05/2026` (`%d` to dzień, `%m` miesiąc, `%Y` rok). Znaczniki, których funkcja nie zna, mają zostać w tekście bez zmian.

Żeby wynik nie zależał od tego, kiedy uruchamiamy test, w testach „zatrzymaliśmy zegar” na 6 maja 2026, godz. 15:42:07 (środa). Notatka, dla której wypełniamy szablon, to plik `Note.md` w folderze `projects/sub`.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B17.1 | z dokumentacji | `{{date}}`, `{{time}}`, `{{datetime}}` | `2026-05-06`, `15:42:07`, `2026-05-06 15:42:07` |  |
| B17.2 | z dokumentacji | `{{year}}`, `{{month}}`, `{{day}}` | `2026`, `05`, `06` | miesiąc i dzień zawsze dwiema cyframi |
| B17.3 | z dokumentacji | `{{title}}`, `{{folder}}` | `Note`, `sub` | nazwa pliku bez `.md` i folder, w którym bezpośrednio leży |
| B17.4 | poprawne | kilka znaczników w jednym tekście: `# {{title}} on {{date}}` | `# Note on 2026-05-06` |  |
| B17.5 | poprawne | ten sam znacznik dwa razy: `{{day}} {{day}}` | `06 06` |  |
| B17.6 | graniczne | tekst bez znaczników, pusty tekst | tekst bez zmian |  |
| B17.7 | z dokumentacji | `{{timestamp}}` | liczba sekund odpowiadająca zatrzymanej dacie |  |
| B17.8 | graniczne | `{{folder}}` dla notatki leżącej w głównym folderze (`Note.md`), w jednym folderze (`a/Note.md`) i w dwóch (`a/b/Note.md`) | `Root`, `a`, `b` | dla głównego folderu wychodzi `Root` |
| B17.9 | z dokumentacji | własny zapis daty: `{{datetime:%Y%m%d%H%M%S}}`, `{{date:%d/%m/%Y}}` | `20260506154207`, `06/05/2026` |  |
| B17.10 | poprawne | inne własne zapisy: nazwa dnia tygodnia `{{date:%A}}`, godzina bez dwukropków `{{time:%H%M%S}}`, numer tygodnia w roku `{{date:%V}}` | `Wednesday`, `154207`, `19` |  |
| B17.11 | błędne | własny zapis z nieistniejącym kodem: `{{date:%Q}}` | znacznik zostaje bez zmian |  |
| B17.12 | błędne | nieznany znacznik `{{foo}}`; spacje w środku `{{ date }}`; pojedyncze nawiasy `{date}`; nieznany znacznik z własnym zapisem `{{foo:%Y}}` | tekst bez zmian |  |

In [121]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders tests/unit/blackbox/test_tags_templates_links.py::test_template_timestamp tests/unit/blackbox/test_tags_templates_links.py::test_template_folder tests/unit/blackbox/test_tags_templates_links.py::test_template_custom_format tests/unit/blackbox/test_tags_templates_links.py::test_template_invalid_format_left_unchanged tests/unit/blackbox/test_tags_templates_links.py::test_template_unknown_placeholder_untouched -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 26 items

tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{date}}-2026-05-06] PASSED [  3%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{time}}-15:42:07] PASSED [  7%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{datetime}}-2026-05-06 15:42:07] PASSED [ 11%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{year}}-2026] PASSED [ 15%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{month}}-05] PASSED [ 19%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{day}}-06] PASSED [ 23%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_placeholders[{{title}}-Note] PASSED [ 26%]
tests/unit/blackbox/test_tags_templates_links.py::test_template_named_place

### B18. convert_wikilinks_to_html (backend/export.py)

W notatce można zrobić link do innej notatki, wpisując jej nazwę w podwójnych nawiasach kwadratowych, np. `[[Zakupy]]`. Taki zapis to **wikilink**. Po pionowej kresce można podać tekst, który ma się wyświetlić zamiast nazwy: `[[Zakupy|lista zakupów]]`.

Notatkę można wyeksportować do HTML (czyli zapisać jako stronę internetową). Ta funkcja zamienia przy tym wikilinki na zwykłe linki HTML. Wyjątkiem są fragmenty oznaczone jako kod: tekst w odwrotnych apostrofach (`` `tak` ``) albo cały blok kodu między liniami z trzema odwrotnymi apostrofami. Tam wikilinki mają zostać tak, jak są.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B18.1 | z dokumentacji | `'see [[Note]] here'` | `[[Note]]` zamienione na link HTML (z klasą `wikilink`, tytułem `Note` i tekstem `Note`), reszta zdania bez zmian |  |
| B18.2 | z dokumentacji | `'[[Note\|shown text]]'` | link prowadzi do `Note`, a wyświetla się `shown text` |  |
| B18.3 | poprawne | `'[[Note#Section]]'` | link prowadzi do `Note#Section` | link do konkretnego nagłówka w notatce |
| B18.4 | poprawne | `'[[ Note \| shown ]]'` | link do `Note`, wyświetla się `shown` | spacje na początku i końcu są usuwane |
| B18.5 | poprawne | `'[[A]] and [[B]]'` | 2 osobne linki |  |
| B18.6 | błędne | tekst bez wikilinków, pusty tekst, `![[image.png]]` | bez zmian | `![[...]]` z wykrzyknikiem oznacza wstawienie obrazka, a nie link |
| B18.7 | z dokumentacji | wikilink wewnątrz kodu: w jednej linii i w bloku kodu | bez zmian |  |
| B18.8 | poprawne | jeden tekst, a w nim: kod z wikilinkiem, zwykły wikilink i blok kodu z wikilinkiem | link powstaje tylko ze zwykłego wikilinka |  |
| B18.9 | błędne | wikilink, w którego nazwie jest kod HTML (`<script>`, `<b>`) | w wyniku nie ma działających znaczników HTML | sprawdzamy, czy nie da się w ten sposób przemycić na stronę złośliwego skryptu |

In [122]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_plain tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_alias tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_heading tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_surrounding_spaces_trimmed tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_several_in_one_text tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_left_literal tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_converted_outside_code_only tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_html_is_escaped -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 13 items

tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_plain PASSED [  7%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_alias PASSED [ 15%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_heading PASSED [ 23%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_surrounding_spaces_trimmed PASSED [ 30%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_several_in_one_text PASSED [ 38%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_left_literal[no links at all] PASSED [ 46%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_left_literal[] PASSED [ 53%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_left_literal[![[image.png]]] PASSED [ 61%]
tests/unit/blackbox/test_tags_templates_links.py::test_wikilink_left_literal[`[[Note]]`] PASSED [ 69%]
tests/unit/blackb

### B19. extract_links_from_content (backend/note_index.py)

Funkcja czyta treść notatki i wypisuje, do czego ta notatka linkuje. Zwraca dwie listy:
- nazwy notatek z wikilinków, czyli zapisów `[[Nazwa]]` (patrz B18),
- adresy ze zwykłych linków Markdown, czyli zapisów `[tekst](plik.md)`.

Na tej podstawie aplikacja pokazuje, które notatki linkują do danej notatki (tzw. backlinki), i rysuje graf powiązań między notatkami.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B19.1 | graniczne | pusty tekst, tekst bez linków | obie listy puste |  |
| B19.2 | poprawne | `'[[Note]]'`, `'[[A]] and [[B]]'` | `Note`; `A`, `B` |  |
| B19.3 | poprawne | `'[[Note\|alias]]'` | `Note` | liczy się nazwa notatki, a nie tekst wyświetlany po kresce |
| B19.4 | poprawne | `'[[  Note  ]]'` | `Note` | spacje usunięte |
| B19.5 | poprawne | `'[[Note#Section]]'` | `Note#Section` | link do nagłówka w notatce |
| B19.6 | poprawne | linki Markdown: `'[text](other.md)'`, `'[text](sub/other.md)'` | na liście linków Markdown: `other.md`; `sub/other.md` |  |
| B19.7 | poprawne | `'[[A]] [text](b.md)'` | wikilinki: `A`, linki Markdown: `b.md` | oba rodzaje w jednym tekście |
| B19.8 | błędne | linki do stron internetowych (`http://`, `https://`), adres e-mail (`mailto:`), link do miejsca w tej samej notatce (`#section`), dane wpisane w link (`data:`) | lista linków Markdown pusta | założyliśmy, że liczą się tylko linki do innych plików z notatkami |

In [123]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_tags_templates_links.py::test_extract_links tests/unit/blackbox/test_tags_templates_links.py::test_extract_links_ignores_non_note_targets -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 15 items

tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[-wikilinks0-mdlinks0] PASSED [  6%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[plain text-wikilinks1-mdlinks1] PASSED [ 13%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[[Note]]-wikilinks2-mdlinks2] PASSED [ 20%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[[A]] and [[B]]-wikilinks3-mdlinks3] PASSED [ 26%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[[Note|alias]]-wikilinks4-mdlinks4] PASSED [ 33%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[[  Note  ]]-wikilinks5-mdlinks5] PASSED [ 40%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[[Note#Section]]-wikilinks6-mdlinks6] PASSED [ 46%]
tests/unit/blackbox/test_tags_templates_links.py::test_extract_links[[text]

### B20. extract_search_terms (backend/note_index.py)

Żeby szybko wyszukiwać notatki, aplikacja dzieli ich treść na pojedyncze słowa i je zapamiętuje (to jest tzw. indeks wyszukiwania). Ta funkcja robi ten podział: dostaje tekst i zwraca zbiór słów (każde słowo raz, kolejność nie ma znaczenia). W `FEATURES.md` jest napisane, że wyszukiwanie nie rozróżnia wielkich i małych liter, więc słowa powinny być zapisane małymi literami.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B20.1 | graniczne | pusty tekst, tekst z samych znaków interpunkcyjnych | brak słów |  |
| B20.2 | poprawne | `'Hello WORLD'` | `hello`, `world` | zamiana na małe litery |
| B20.3 | poprawne | `'hello,world.again!'` | `hello`, `world`, `again` | przecinek, kropka itp. rozdzielają słowa, nawet bez spacji |
| B20.4 | poprawne | `'Word word WORD'` | `word` | to samo słowo tylko raz |
| B20.5 | graniczne | `'a b go'` | `go` | słowa jednoliterowe są pomijane, dwuliterowe już nie |
| B20.6 | poprawne | `'foo-bar foo_bar'` | `foo-bar`, `foo_bar` | myślnik i podkreślnik nie dzielą słowa |
| B20.7 | poprawne | `'v2 2026'` | `v2`, `2026` | cyfry też są częścią słów |
| B20.8 | poprawne | `'zażółć'`, `'café'` | `za`; `caf` | litery spoza alfabetu angielskiego (np. polskie) przerywają słowo. Tak funkcja działa obecnie |

In [124]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_tags_templates_links.py::test_search_terms tests/unit/blackbox/test_tags_templates_links.py::test_search_terms_only_ascii_runs_are_indexed -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 12 items

tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[-expected0] PASSED [  8%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[!!! ... ???-expected1] PASSED [ 16%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[Hello WORLD-expected2] PASSED [ 25%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[hello,world.again!-expected3] PASSED [ 33%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[line one\nline two-expected4] PASSED [ 41%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[Word word WORD-expected5] PASSED [ 50%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[a b go-expected6] PASSED [ 58%]
tests/unit/blackbox/test_tags_templates_links.py::test_search_terms[foo-bar foo_bar-expected7] PASSED [ 66%]
tests/unit/blackbox/test_tags_templates_

### B21. parse_theme_metadata (backend/themes.py)

Wygląd aplikacji można zmieniać **motywami**. Motyw to plik CSS, czyli plik opisujący kolory i style. Na początku takiego pliku może być komentarz, który mówi, czy motyw jest jasny, czy ciemny: `/* @theme-type: light */` (jasny) albo `/* @theme-type: dark */` (ciemny) – opisane w `THEMES.md`.

Funkcja czyta ten komentarz i zwraca rodzaj motywu w postaci `{'type': 'light'}` albo `{'type': 'dark'}`. Jeśli komentarza nie ma albo jest niepoprawny, przyjmuje, że motyw jest ciemny (`dark`).

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B21.1 | z dokumentacji | komentarz `/* @theme-type: light */` w pierwszej linii | `light` |  |
| B21.2 | z dokumentacji | komentarz `/* @theme-type: dark */` w pierwszej linii | `dark` |  |
| B21.3 | poprawne | komentarz `light` w drugiej linii | `light` | nie musi być w pierwszej linii |
| B21.4 | błędne | plik bez komentarza, pusty plik | `dark` | wartość domyślna |
| B21.5 | błędne | nieznany rodzaj: `@theme-type: blue` | `dark` | zła wartość, więc domyślna |
| B21.6 | graniczne | komentarz `light` dopiero po 20 pustych liniach | `dark` | funkcja szuka komentarza tylko na początku pliku, a to już nie jest początek |
| B21.7 | błędne | plik nie istnieje | `dark` | bez błędu |

In [125]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_missing_file_returns_default -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 8 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[/* @theme-type: light */\nbody {}-light] PASSED [ 12%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[/* @theme-type: dark */\nbody {}-dark] PASSED [ 25%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[/* My theme */\n/* @theme-type: light */\nbody {}-light] PASSED [ 37%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[body {}-dark] PASSED [ 50%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[-dark] PASSED [ 62%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[/* @theme-type: blue */-dark] PASSED [ 75%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_theme_type[\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n/* @theme-type: light */-dark] PASSED [ 87%]
tests/unit/blackbox/test_auth_th

### B22. verify_api_key (backend/main.py)

Inne programy (np. skrypty albo agent AI) mogą łączyć się z aplikacją bez logowania, podając **klucz API**. Właściwy klucz jest zapisany w ustawieniach aplikacji. Funkcja porównuje klucz podany przez program z kluczem z ustawień. Zwraca `True` (wpuść) tylko wtedy, gdy są dokładnie takie same, a w każdym innym przypadku `False` (nie wpuszczaj).


| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B22.1 | błędne | w ustawieniach nie ustawiono klucza; program podaje `anything` albo pusty klucz | False | przyjęliśmy, że gdy klucza nie ustawiono, nikt nie może wejść przez klucz |
| B22.2 | błędne | w ustawieniach w ogóle nie ma części dotyczącej logowania | False |  |
| B22.3 | graniczne | `secret` vs pusty klucz | False |  |
| B22.4 | błędne | `secret` vs `wrong` | False | zły klucz |
| B22.5 | graniczne | `secret` vs `secre` | False | sam początek klucza nie wystarczy |
| B22.6 | błędne | `secret` vs `SECRET` | False | wielkość liter ma znaczenie |
| B22.7 | poprawne | `secret` vs `secret` | True | dobry klucz |
| B22.8 | poprawne | `sekret-ąę` vs `sekret-ąę` | True | klucz z polskimi znakami też działa |
| B22.9 | błędne | `secret` vs `sekret-ąę` | False, program nie zgłasza błędu | sprawdzamy, czy polskie znaki w złym kluczu nie psują porównania |

In [126]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key_without_auth_settings -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 10 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[-anything-False] PASSED [ 10%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[--False] PASSED [ 20%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[secret--False] PASSED [ 30%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[secret-wrong-False] PASSED [ 40%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[secret-secre-False] PASSED [ 50%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[secret-SECRET-False] PASSED [ 60%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[secret-secret-True] PASSED [ 70%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_api_key[sekret-\u0105\u0119-sekret-\u0105\u0119-True] PASSED [ 80%]
tests

### B23. verify_password (backend/main.py)

Funkcja sprawdza hasło wpisane przy logowaniu. Aplikacja nie przechowuje samego hasła, tylko jego hash. Z hasha nie da się odczytać hasła, ale da się sprawdzić, czy wpisane hasło do niego pasuje (aplikacja używa do tego algorytmu bcrypt, `AUTHENTICATION.md`). Funkcja zwraca `True`, gdy hasło jest dobre, i `False`, gdy złe.


| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B23.1 | poprawne | `pass` vs `pass` | True |  |
| B23.2 | błędne | `pass` vs `wrong` | False |  |
| B23.3 | graniczne | `pass` vs puste hasło | False |  |
| B23.4 | poprawne | `hasło-ąę` vs `hasło-ąę` | True | polskie znaki |
| B23.5 | błędne | `pass` vs `hasło-ąę` | False |  |
| B23.6 | błędne | w ustawieniach nie ma hasha (hasła nie ustawiono) | False | nie da się zalogować |
| B23.7 | błędne | hash w ustawieniach jest uszkodzony | False, program nie zgłasza błędu |  |

In [127]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password_no_hash_configured tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password_malformed_hash -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 8 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password[pass-pass-True] PASSED [ 12%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password[pass-wrong-False] PASSED [ 25%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password[pass--False] PASSED [ 37%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password[has\u0142o-\u0105\u0119-has\u0142o-\u0105\u0119-True] PASSED [ 50%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password[pass-has\u0142o-\u0105\u0119-False] PASSED [ 62%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password_no_hash_configured PASSED [ 75%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify_password_malformed_hash[not-a-bcrypt-hash] PASSED [ 87%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_verify

### B24. Plugin._open_tasks (plugins/contrib/search_open_tasks.py)

NoteDiscovery ma **wtyczki**, czyli dodatki rozszerzające aplikację. Jedna z nich szuka w notatkach zadań, które nie zostały jeszcze zrobione. W Markdownie zadanie to punkt listy z kwadratowym nawiasem: `- [ ] kupić mleko` to zadanie do zrobienia, a `- [x] kupić mleko` to zadanie zrobione (odhaczone).

Według opisu funkcji liczą się zadania w zwykłych listach (zaczynające się od `-`, `*` albo `+`) i w listach numerowanych (`1. [ ]`, `1) [ ]`). Zapis `- []` (bez spacji w nawiasie) nie jest zadaniem. Funkcja zwraca listę znalezionych zadań: numer linii i treść zadania.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B24.1 | z dokumentacji | `- [ ] todo`, `* [ ] todo`, `+ [ ] todo` | 1 zadanie | trzy rodzaje punktorów |
| B24.2 | z dokumentacji | `1. [ ] todo`, `1) [ ] todo` | 1 zadanie | listy numerowane |
| B24.3 | poprawne | zadanie przesunięte w prawo (spacjami albo tabulatorem) | 1 zadanie | np. zadanie podrzędne |
| B24.4 | błędne | pusty tekst, zwykły tekst, zwykły punkt listy bez `[ ]` | brak zadań |  |
| B24.5 | błędne | zadania odhaczone: `- [x] done`, `- [X] done` | brak zadań | zrobione zadania są pomijane |
| B24.6 | z dokumentacji | `- [] no space in the box` | brak zadań | brak spacji w nawiasie |
| B24.7 | graniczne | `- [ ]` bez żadnego tekstu | brak zadań | puste zadanie się nie liczy |
| B24.8 | błędne | `[ ] no list marker` | brak zadań | brak `-`, `*` itp. na początku |
| B24.9 | poprawne | notatka: nagłówek, 2 zadania do zrobienia i 1 zrobione | 2 zadania, z linii 2 i 5 | sprawdzamy też numery linii |
| B24.10 | błędne | zadanie z kodem HTML `<b>` w treści | `<b>` zamienione na `&lt;b&gt;` | wynik jest pokazywany na stronie, więc HTML trzeba „rozbroić”, żeby przeglądarka pokazała go jako zwykły tekst |
| B24.11 | graniczne | treść zadania ma 200 albo 201 znaków | 200: cała treść; 201: pierwsze 200 znaków i `…` | za długie treści są skracane |

In [128]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_not_found tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_tasks_only_unchecked_with_line_numbers tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_text_is_escaped tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_long_text_is_truncated -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 19 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[- [ ] todo] PASSED [  5%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[* [ ] todo] PASSED [ 10%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[+ [ ] todo] PASSED [ 15%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[1. [ ] todo] PASSED [ 21%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[1) [ ] todo] PASSED [ 26%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[  - [ ] todo] PASSED [ 31%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_found[\t- [ ] todo] PASSED [ 36%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task_not_found[] PASSED [ 42%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_open_task

### B25. validate_slug (backend/share.py)

Notatkę można udostępnić innym przez link. Zamiast losowego ciągu znaków (patrz B26) można wymyślić własną końcówkę linku, np. `moja-notatka`. Taka końcówka to **slug**. Funkcja sprawdza, czy slug jest poprawny. Według `SHARING.md` może zawierać tylko litery alfabetu angielskiego (bez polskich znaków), cyfry, `-` i `_`, i musi mieć od 3 do 64 znaków. Spacje na początku i na końcu są usuwane.

Jeśli slug jest poprawny, funkcja go zwraca. Jeśli nie, zgłasza błąd `ShareSlugError` z powodem: `too_short` (za krótki), `too_long` (za długi) albo `invalid_chars` (niedozwolone znaki).

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B25.1 | poprawne | `My_note-1`, `ABC`, `123` | zwrócony bez zmian |  |
| B25.2 | graniczne | slug z 3 znaków i z 64 znaków | zwrócony bez zmian | najkrótszy i najdłuższy dozwolony |
| B25.3 | graniczne | 2 znaki | błąd `too_short` | o jeden znak za krótki |
| B25.4 | graniczne | 65 znaków | błąd `too_long` | o jeden znak za długi |
| B25.5 | poprawne | `abc` ze spacjami, tabulatorem albo enterem przed i po | `abc` | białe znaki na brzegach są usuwane |
| B25.6 | graniczne | 64 znaki ze spacjami na brzegach; `'  ab  '` | 64 znaki; błąd `too_short` | długość liczy się dopiero po usunięciu spacji |
| B25.7 | błędne | pusty tekst, same spacje | błąd `too_short` |  |
| B25.8 | błędne | zamiast tekstu: `None` (brak wartości), liczba 123, lista | błąd `invalid_chars` | to w ogóle nie jest tekst |
| B25.9 | błędne | w środku sluga: spacja, `/`, `\`, `.`, `?` albo enter | błąd `invalid_chars` | 6 przypadków |
| B25.10 | błędne | `ąbc` | błąd `invalid_chars` | polska litera |

In [129]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_rejected tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_whitespace_trimmed -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 23 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged[abc] PASSED [  4%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged[aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa] PASSED [  8%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged[My_note-1] PASSED [ 13%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged[ABC] PASSED [ 17%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_valid_returned_unchanged[123] PASSED [ 21%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_rejected[None-invalid_chars] PASSED [ 26%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_slug_rejected[123-invalid_chars] PASSED [ 30%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::tes

### B26. generate_token (backend/share.py)

Przy udostępnianiu notatki aplikacja tworzy **token**, czyli losowy ciąg znaków, który staje się częścią linku do notatki. Dzięki temu linku nie da się zgadnąć. W `SHARING.md` jest napisane, że token ma 16 znaków. Funkcji można też podać inną długość.

| Nr | Rodzaj | Dane wejściowe | Oczekiwany wynik | Uwagi |
|---|---|---|---|---|
| B26.1 | z dokumentacji | bez podania długości | token ma 16 znaków |  |
| B26.2 | graniczne | długość 0, 1, 32 | token ma podaną długość |  |
| B26.3 | poprawne | długość 500 | same litery, cyfry, `-` i `_` | długi token, żeby sprawdzić dużo znaków naraz; takie znaki można bezpiecznie wstawić do linku |
| B26.4 | poprawne | dwa wywołania funkcji | dwa różne tokeny | token ma być losowy |

In [130]:
!cd ../.. && python -m pytest tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_default_length tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_requested_length tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_is_url_safe tests/unit/blackbox/test_auth_themes_plugins_share.py::test_tokens_differ -v -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

============================= test session starts =============================
collecting ... collected 6 items

tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_default_length PASSED [ 16%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_requested_length[0] PASSED [ 33%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_requested_length[1] PASSED [ 50%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_requested_length[32] PASSED [ 66%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_token_is_url_safe PASSED [ 83%]
tests/unit/blackbox/test_auth_themes_plugins_share.py::test_tokens_differ PASSED [100%]

============================== 6 passed in 0.03s ==============================


## Uruchomienie testów

Z głównego folderu repozytorium:

```
pip install -e .[dev]
pytest tests/unit/blackbox
```

In [131]:
!cd ../.. && python -m pytest tests/unit/blackbox -q -rx --no-header -p no:warnings --color=yes -p no:cacheprovider -W ignore::DeprecationWarning

........................................................................ [ 30%]
..........................................xx............................ [ 60%]
........................................................................ [ 90%]
.......................                                                  [100%]
=========================== short test summary info ===========================
XFAIL tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_quoted_values[tags: ["a", "b"]] - Bug: YAML quotes are kept as part of the tag name
XFAIL tests/unit/blackbox/test_tags_templates_links.py::test_parse_tags_quoted_values[tags: ['a', 'b']] - Bug: YAML quotes are kept as part of the tag name
237 passed, 2 xfailed in 0.43s


## Wnioski

Przechodzą wszystkie testy oprócz dwóch, które oznaczyliśmy jako `xfail`.

Znaleziony błąd (B16.13): `parse_tags` nie usuwa cudzysłowów z tagów. Dla `tags: ["a", "b"]` zwraca tagi razem z cudzysłowami, a nie `a` i `b`.
Taki zapis jest poprawny w YAML-u, więc użytkownik może tak napisać, a potem w aplikacji ten sam tag pojawia się dwa razy.
Test zostawiliśmy z `xfail(strict=True)`. Jak ktoś naprawi błąd, test zacznie przechodzić i pytest da znać, że trzeba zdjąć oznaczenie.

Miejsca, w których dokumentacja i działanie funkcji się nie zgadzają albo dokumentacja nic nie mówi:
- B11.1: z dokumentacji wynika, że pusta notatka ma 0 minut czytania, a funkcja zwraca 1,
- B12.4: docstring `load_config` mówi, że przy złej konfiguracji leci `ValueError`, a przy złym timeoucie funkcja po cichu bierze wartość domyślną,
- B1.6 i B1.7: nie wiadomo, co ma się dziać przy ujemnym offsecie i limicie 0.